# Лабораторная работа №4. Анализ систем синтеза речи

Ноутбук повторяет вычислительную часть отчёта: читает CSV с разметкой, считает задержки и проценты ошибок, а также проверяет наличие сохранённых WAV-файлов для пяти TTS из плана.

In [1]:
import csv
from html import escape
from pathlib import Path

try:
    from IPython.display import Audio, HTML, display
except ModuleNotFoundError:
    Audio = None
    HTML = None
    display = None

LAB_DIR = Path.cwd() if Path.cwd().name == 'lab4' else Path.cwd() / 'lab4'
DATA_DIR = LAB_DIR / 'data'
AUDIO_DIR = LAB_DIR / 'audio'
SYSTEMS = ['silero', 'rhvoice', 'piper', 'coqui_xtts', 'espeak_ng']
SYSTEM_NAMES = ['Silero TTS', 'RHVoice', 'Piper', 'Coqui XTTS', 'eSpeak NG']
SYSTEM_LABELS = dict(zip(SYSTEMS, SYSTEM_NAMES))
LAB_DIR

PosixPath('/Users/alice/Documents/ITMO/2_sem/OZiR/lab4')

In [2]:
def read_csv(path):
    with path.open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))

latency_rows = read_csv(DATA_DIR / 'latency.csv')
results_rows = read_csv(DATA_DIR / 'results.csv')
manifest_rows = read_csv(AUDIO_DIR / 'manifest.csv')
len(latency_rows), len(results_rows), len(manifest_rows)

(25, 150, 155)

In [3]:
from statistics import mean
from collections import defaultdict

latency = defaultdict(list)
for row in latency_rows:
    latency[row['system']].append(float(row['latency_seconds']))

for system in SYSTEM_NAMES:
    values = latency[system]
    print(f'{system:12s} mean={mean(values):.2f}s min={min(values):.2f}s max={max(values):.2f}s')

Silero TTS   mean=0.72s min=0.69s max=0.76s
RHVoice      mean=0.22s min=0.20s max=0.24s
Piper        mean=0.30s min=0.28s max=0.32s
Coqui XTTS   mean=1.42s min=1.36s max=1.48s
eSpeak NG    mean=0.12s min=0.11s max=0.14s


In [4]:
quality = defaultdict(lambda: {'targets': 0, 'errors': 0})
for row in results_rows:
    key = (row['system'], row['category'])
    quality[key]['targets'] += int(row['target_count'])
    quality[key]['errors'] += int(row['error_count'])

for category in ['graph_abbrev', 'initial_abbrev', 'numbers']:
    print('\n' + category)
    for system in SYSTEM_NAMES:
        item = quality[(system, category)]
        percent = 100 * item['errors'] / item['targets']
        print(f'{system:12s} {item["errors"]:2d}/{item["targets"]:2d} = {percent:5.2f}%')


graph_abbrev
Silero TTS    1/18 =  5.56%
RHVoice       2/18 = 11.11%
Piper         3/18 = 16.67%
Coqui XTTS    7/18 = 38.89%
eSpeak NG    14/18 = 77.78%

initial_abbrev
Silero TTS    5/18 = 27.78%
RHVoice       6/18 = 33.33%
Piper         7/18 = 38.89%
Coqui XTTS   12/18 = 66.67%
eSpeak NG    16/18 = 88.89%

numbers
Silero TTS    4/30 = 13.33%
RHVoice       5/30 = 16.67%
Piper         8/30 = 26.67%
Coqui XTTS   13/30 = 43.33%
eSpeak NG    22/30 = 73.33%


In [5]:
for engine in SYSTEMS:
    files = sorted((AUDIO_DIR / engine).rglob('*.wav'))
    print(f'{engine:10s}: {len(files)} wav files')
    assert len(files) == 31, f'{engine}: expected 31 files, got {len(files)}'

assert len(manifest_rows) == 155
print('Audio manifest is complete.')

silero    : 31 wav files
rhvoice   : 31 wav files
piper     : 31 wav files
coqui_xtts: 31 wav files
espeak_ng : 31 wav files
Audio manifest is complete.


## Прослушивание примеров для презентации

В этой секции используются уже сгенерированные аудиофайлы из `audio/manifest.csv`. Для каждого выбранного предложения выводятся пять плееров: Silero TTS, RHVoice, Piper, Coqui XTTS и eSpeak NG. Так можно быстро включить один и тот же текст в разных синтезаторах и сравнить качество на слух во время презентации.

In [ ]:
sentence_rows = read_csv(DATA_DIR / 'test_sentences.csv')
sentences_by_key = {
    (row['category'], int(row['item'])): row
    for row in sentence_rows
}
manifest_by_key = {
    (row['engine'], row['category'], int(row['item'])): LAB_DIR / row['audio_path']
    for row in manifest_rows
}

DEMO_EXAMPLES = [
    ('graph_abbrev', 1, 'Графические сокращения: адрес'),
    ('initial_abbrev', 3, 'Аббревиатуры: БД и ЛВС'),
    ('numbers', 7, 'Цифровые обозначения: дроби'),
    ('stress', 1, 'Ударение и естественность речи'),
]


def show_audio_comparison(category, item, title):
    sentence = sentences_by_key[(category, item)]
    if Audio is None:
        print(f'IPython is not installed, audio widgets are unavailable: {title}')
        print(f'Text: {sentence["text"]}')
        for engine in SYSTEMS:
            audio_path = manifest_by_key[(engine, category, item)]
            if not audio_path.exists():
                raise FileNotFoundError(f'Audio file is missing: {audio_path}')
            print(f'{SYSTEM_LABELS[engine]}: {audio_path.relative_to(LAB_DIR)}')
        return

    display(HTML(f'<h3>{escape(title)}</h3>'))
    display(HTML(f'<p><b>Текст:</b> {escape(sentence["text"])}</p>'))
    display(HTML(f'<p><b>Проверяемые элементы:</b> {escape(sentence["targets"])}</p>'))

    for engine in SYSTEMS:
        audio_path = manifest_by_key[(engine, category, item)]
        if not audio_path.exists():
            raise FileNotFoundError(f'Audio file is missing: {audio_path}')
        display(HTML(f'<b>{escape(SYSTEM_LABELS[engine])}</b> <code>{escape(str(audio_path.relative_to(LAB_DIR)))}</code>'))
        display(Audio(filename=str(audio_path)))


for category, item, title in DEMO_EXAMPLES:
    show_audio_comparison(category, item, title)